# Shopify Competitor Price Monitor

Automated price monitoring for Shopify-based e-commerce competitors.

## How It Works
1. Shopify stores expose a public `/products.json` API — no scraping needed
2. Fetch product catalogs from multiple competitors
3. Store price snapshots in SQLite for historical comparison
4. Detect price changes and send Telegram alerts

**Tech Stack:** Python, Requests, Pandas, SQLite, Shopify API, Telegram Bot

In [ ]:
import requests
import json
import sqlite3
import time
import pandas as pd
from datetime import datetime
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================

# Shopify stores to monitor (name, base_url)
STORES = [
    ("ColourPop", "https://colourpop.com"),
    ("Allbirds", "https://www.allbirds.com"),
    ("Matt & Nat", "https://mattandnat.com"),
]

# Price change threshold (%) to trigger alert
PRICE_CHANGE_THRESHOLD = 5.0

# Telegram config (leave empty to disable)
TELEGRAM_BOT_TOKEN = ""
TELEGRAM_CHAT_ID = ""

print(f"Monitoring {len(STORES)} stores | Threshold: {PRICE_CHANGE_THRESHOLD}%")

## 1. Shopify API: Fetch Products

Shopify stores expose `/products.json` publicly. We can fetch up to 250 products per page.

In [ ]:
def fetch_products(base_url, page=1, limit=250):
    """Fetch a page of products from a Shopify store."""
    url = f"{base_url}/products.json"
    try:
        resp = requests.get(url, params={"limit": limit, "page": page}, timeout=30)
        resp.raise_for_status()
        return resp.json().get("products", [])
    except Exception as e:
        print(f"  Error: {e}")
        return []

def fetch_all_products(base_url, max_pages=2):
    """Fetch all products from a store (up to max_pages)."""
    all_products = []
    for page in range(1, max_pages + 1):
        products = fetch_products(base_url, page)
        if not products:
            break
        all_products.extend(products)
        print(f"  Page {page}: {len(products)} products")
        if len(products) < 250:
            break
        time.sleep(1)  # Rate limiting
    return all_products

# Test with one store
test_products = fetch_products("https://colourpop.com")
if test_products:
    p = test_products[0]
    print(f"Sample product: {p['title']}")
    print(f"  Vendor: {p['vendor']}")
    print(f"  Variants: {len(p['variants'])}")
    print(f"  Price: ${p['variants'][0]['price']}")

## 2. Database Layer (SQLite)

Store product snapshots and detect price changes over time.

In [ ]:
DB_PATH = "data/prices.db"
Path("data").mkdir(exist_ok=True)

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""CREATE TABLE IF NOT EXISTS price_snapshots (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        store TEXT NOT NULL,
        product_id INTEGER NOT NULL,
        product_title TEXT NOT NULL,
        variant_id INTEGER NOT NULL,
        variant_title TEXT,
        price REAL NOT NULL,
        compare_at_price REAL,
        available INTEGER,
        snapshot_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )""")
    conn.execute("""CREATE TABLE IF NOT EXISTS price_changes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        store TEXT NOT NULL,
        product_title TEXT NOT NULL,
        variant_title TEXT,
        old_price REAL NOT NULL,
        new_price REAL NOT NULL,
        change_pct REAL NOT NULL,
        direction TEXT NOT NULL,
        detected_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )""")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_lookup ON price_snapshots(store, product_id, variant_id, snapshot_at DESC)")
    conn.commit()
    return conn

conn = init_db()
print("Database initialized.")

In [ ]:
def get_last_price(conn, store, product_id, variant_id):
    """Get the most recent price for a variant."""
    cur = conn.execute(
        "SELECT price FROM price_snapshots WHERE store=? AND product_id=? AND variant_id=? ORDER BY snapshot_at DESC LIMIT 1",
        (store, product_id, variant_id)
    )
    row = cur.fetchone()
    return row[0] if row else None

def save_snapshot(conn, store, pid, title, vid, vtitle, price, compare_price, available):
    conn.execute(
        "INSERT INTO price_snapshots (store,product_id,product_title,variant_id,variant_title,price,compare_at_price,available) VALUES (?,?,?,?,?,?,?,?)",
        (store, pid, title, vid, vtitle, price, compare_price, available)
    )

print("DB helper functions ready.")

## 3. Telegram Alert

Send real-time notifications when prices change beyond the threshold.

In [ ]:
def send_telegram_alert(product_title, variant_title, old_price, new_price, change_pct, direction, store):
    """Send price change alert via Telegram."""
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        # Print to console if Telegram not configured
        emoji = "\U0001f4c9" if direction == "decrease" else "\U0001f4c8"
        print(f"  {emoji} {product_title}: ${old_price:.2f} -> ${new_price:.2f} ({change_pct:+.1f}%)")
        return
    
    emoji = "\U0001f4c9" if direction == "decrease" else "\U0001f4c8"
    msg = (
        f"{emoji} Price {direction}\n\n"
        f"Store: {store}\n"
        f"Product: {product_title}\n"
        f"${old_price:.2f} -> ${new_price:.2f} ({change_pct:+.1f}%)"
    )
    try:
        requests.post(
            f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage",
            json={"chat_id": TELEGRAM_CHAT_ID, "text": msg}, timeout=10
        )
    except Exception as e:
        print(f"  Telegram error: {e}")

print("Telegram alert function ready.")

## 4. Run the Monitor

Fetch all products, compare against last snapshot, detect changes.

In [ ]:
def run_monitor():
    """Main monitoring cycle."""
    print(f"\n{'='*60}")
    print(f"Shopify Price Monitor - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*60}\n")
    
    all_changes = []
    total_products = 0
    
    for store_name, base_url in STORES:
        print(f"[{store_name}] Fetching from {base_url}...")
        products = fetch_all_products(base_url)
        if not products:
            print(f"  No products found.")
            continue
        
        print(f"  Found {len(products)} products")
        changes = 0
        
        for product in products:
            pid = product["id"]
            title = product["title"]
            
            for v in product.get("variants", []):
                vid = v["id"]
                vtitle = v.get("title", "Default")
                price = float(v["price"])
                compare_price = float(v["compare_at_price"]) if v.get("compare_at_price") else None
                available = int(v.get("available", False))
                
                # Check for price change
                last_price = get_last_price(conn, store_name, pid, vid)
                if last_price and last_price > 0:
                    pct = ((price - last_price) / last_price) * 100
                    if abs(pct) >= PRICE_CHANGE_THRESHOLD:
                        direction = "decrease" if pct < 0 else "increase"
                        all_changes.append({
                            "store": store_name, "product": title,
                            "variant": vtitle, "old": last_price, "new": price, "pct": pct
                        })
                        send_telegram_alert(title, vtitle, last_price, price, pct, direction, store_name)
                        conn.execute(
                            "INSERT INTO price_changes (store,product_title,variant_title,old_price,new_price,change_pct,direction) VALUES (?,?,?,?,?,?,?)",
                            (store_name, title, vtitle, last_price, price, pct, direction)
                        )
                        changes += 1
                
                save_snapshot(conn, store_name, pid, title, vid, vtitle, price, compare_price, available)
        
        conn.commit()
        total_products += len(products)
        print(f"  {changes} price changes detected")
        time.sleep(2)
    
    print(f"\n{'='*60}")
    print(f"Done. Products: {total_products}, Changes: {len(all_changes)}")
    print(f"{'='*60}\n")
    return all_changes

changes = run_monitor()

## 5. Analyse Results with Pandas

In [ ]:
# Product inventory overview
df = pd.read_sql_query("SELECT * FROM price_snapshots ORDER BY snapshot_at DESC LIMIT 5", conn)
print(f"Total snapshots in DB: {pd.read_sql_query('SELECT COUNT(*) as n FROM price_snapshots', conn).iloc[0]['n']}")
print(f"Total products tracked: {pd.read_sql_query('SELECT COUNT(DISTINCT product_id) as n FROM price_snapshots', conn).iloc[0]['n']}")
print(f"\nLatest snapshots:")
df[["store", "product_title", "variant_title", "price", "snapshot_at"]]

In [ ]:
# Price distribution by store
price_stats = pd.read_sql_query("""
    SELECT store, 
           COUNT(DISTINCT product_id) as products,
           ROUND(AVG(price), 2) as avg_price,
           ROUND(MIN(price), 2) as min_price,
           ROUND(MAX(price), 2) as max_price
    FROM price_snapshots 
    GROUP BY store
""", conn)
price_stats

In [ ]:
# Recent price changes (if any)
changes_df = pd.read_sql_query("SELECT * FROM price_changes ORDER BY detected_at DESC LIMIT 20", conn)
if len(changes_df) > 0:
    print(f"Total price changes recorded: {len(changes_df)}")
    changes_df[["store", "product_title", "old_price", "new_price", "change_pct", "direction", "detected_at"]]
else:
    print("No price changes detected yet. Run the monitor again later to detect changes.")

## 6. Visualization

In [ ]:
import matplotlib.pyplot as plt

# Price distribution histogram per store
all_prices = pd.read_sql_query("SELECT store, price FROM price_snapshots", conn)

fig, ax = plt.subplots(figsize=(10, 5))
for store in all_prices["store"].unique():
    store_data = all_prices[all_prices["store"] == store]["price"]
    ax.hist(store_data[store_data < 200], bins=30, alpha=0.5, label=store)
ax.set_xlabel("Price ($)")
ax.set_ylabel("Count")
ax.set_title("Product Price Distribution by Store")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## How to Use

1. **Run once** to establish baseline snapshots
2. **Run again later** (hours/days) to detect price changes
3. **Configure Telegram** in the config cell to get real-time alerts

### Add more stores
Find Shopify stores and add them to the `STORES` list. Any store with `/products.json` accessible works.

### Schedule with cron
```bash
# Run every 6 hours
0 */6 * * * cd /path/to/project && python monitor.py
```